In [ ]:
!pip install rdkit pandas tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 39.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import requests
import time
import re
from rdkit import Chem
from tqdm import tqdm

df = pd.read_csv('smiles.csv')  # must have a 'SMILES' column

#  Step 1: SMILES conversion into InChIKey
def smiles_to_inchikey(smi):
    mol = Chem.MolFromSmiles(smi)
    return Chem.MolToInchiKey(mol) if mol else None

df['inchikey'] = df['SMILES'].apply(smiles_to_inchikey)

#  Step 2: InChIKey -> list of PatentIDs
def get_patent_ids(inchikey, retries=3):
    if inchikey is None:
        return []
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{inchikey}/xrefs/PatentID/JSON"
    for _ in range(retries):
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                return r.json()['InformationList']['Information'][0].get('PatentID', [])
            elif r.status_code == 404:
                return []
            time.sleep(1)
        except requests.exceptions.RequestException:
            time.sleep(1)
    return []

#  Step 3: PatentID -> title (cached, since patents repeat across compounds)
patent_title_cache = {}

def get_patent_title(patent_id, retries=3):
    if patent_id in patent_title_cache:
        return patent_title_cache[patent_id]
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/patent/{patent_id}/JSON"
    title = None
    for _ in range(retries):
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                data = r.json()
                # Title is usually the first Name/Section labeled "Title" in the record
                record = data.get('Record', {})
                title = record.get('RecordTitle')  # PubChem often puts it here directly
                if not title:
                    for section in record.get('Section', []):
                        if section.get('TOCHeading', '').lower() == 'title':
                            info = section.get('Information', [{}])[0]
                            title = info.get('Value', {}).get('StringWithMarkup', [{}])[0].get('String')
                break
            elif r.status_code == 404:
                break
            time.sleep(1)
        except requests.exceptions.RequestException:
            time.sleep(1)
    patent_title_cache[patent_id] = title
    return title

#  Step 4: crude keyword tagging from titles
ACTIVITY_KEYWORDS = ['inhibitor', 'agonist', 'antagonist', 'modulator',
                      'activator', 'blocker', 'inducer', 'degrader', 'binder']

def guess_activity(titles):
    text = ' '.join([t for t in titles if t]).lower()
    hits = sorted(set(kw for kw in ACTIVITY_KEYWORDS if kw in text))
    return ', '.join(hits) if hits else None

#  Run pipeline
MAX_PATENTS_PER_COMPOUND = 3  # cap lookups per compound to keep runtime sane

results = []
for ik in tqdm(df['inchikey']):
    patent_ids = get_patent_ids(ik)
    time.sleep(0.2)

    top_ids = patent_ids[:MAX_PATENTS_PER_COMPOUND]
    titles = []
    for pid in top_ids:
        titles.append(get_patent_title(pid))
        time.sleep(0.2)

    results.append({
        'is_patented': len(patent_ids) > 0,
        'n_patents': len(patent_ids),
        'example_patent_ids': '; '.join(top_ids) if top_ids else None,
        'example_patent_titles': ' | '.join([t for t in titles if t]) if titles else None,
        'guessed_activity': guess_activity(titles)
    })

df = pd.concat([df, pd.DataFrame(results)], axis=1)
df.to_csv('patent_check_results.csv', index=False)
df.head()